In [2]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sqlalchemy import create_engine

engine = create_engine('mysql+pymysql://root:@localhost/dw_agriculture')

# Charger les tables
fait = pd.read_sql("SELECT * FROM fait_meteo", engine)
dim_station = pd.read_sql("SELECT * FROM dim_station", engine)
dim_temps = pd.read_sql("SELECT * FROM dim_temps", engine)
dim_alerte = pd.read_sql("SELECT * FROM dim_alerte", engine)

# Jointures
df = fait.merge(dim_station, on='id_station', how='left')
df = df.merge(dim_temps, on='id_date', how='left')
df = df.merge(dim_alerte, on='id_alerte', how='left')

print("✅ Données chargées :", df.shape)
print(df.columns.tolist())

✅ Données chargées : (4626, 24)
['id_fait', 'id_date', 'id_station', 'id_alerte', 'temp_c', 'hum_pct', 'wind_speed', 'precip_mm', 'severity_index_x', 'indice_risque', 'nom_station', 'ville', 'altitude', 'zone_geo', 'capteur_type', 'date', 'jour', 'mois', 'trimestre', 'annee', 'saison', 'severity_index_y', 'alert_msg', 'type_precip']


In [5]:
# ============================================
# KPI CARDS
# ============================================
total_releves = len(df)
temp_max = df['temp_c'].max()
alertes_rouge = len(df[df['severity_index'] == 'Rouge'])
indice_moyen = round(df['indice_risque'].mean(), 2)
stations_actives = df['id_station'].nunique()

print("=" * 50)
print("📊 DASHBOARD — Agriculture-Résilience 2030")
print("=" * 50)
print(f"📍 Stations actives      : {stations_actives}")
print(f"📋 Total relevés         : {total_releves}")
print(f"🌡️  Température max       : {temp_max}°C")
print(f"🚨 Alertes Rouge         : {alertes_rouge}")
print(f"⚠️  Indice risque moyen   : {indice_moyen}")
print("=" * 50)

# ============================================
# GRAPHIQUE 1 — Température moyenne par zone
# ============================================
temp_zone = df.groupby('zone_geo')['temp_c'].mean().round(1).reset_index()
temp_zone.columns = ['Zone', 'Temp_Moyenne']
temp_zone = temp_zone.sort_values('Temp_Moyenne', ascending=False)

fig1 = px.bar(
    temp_zone,
    x='Zone', y='Temp_Moyenne',
    title='🌡️ Température Moyenne par Zone Géographique',
    color='Temp_Moyenne',
    color_continuous_scale='RdYlGn_r',
    text='Temp_Moyenne'
)
fig1.update_traces(texttemplate='%{text}°C', textposition='outside')
fig1.update_layout(height=450, xaxis_tickangle=-30)
fig1.show()

# ============================================
# GRAPHIQUE 2 — Précipitations par mois
# ============================================
precip_mois = df.groupby('mois')['precip_mm'].sum().reset_index()
precip_mois.columns = ['Mois', 'Precip_Total']
mois_labels = {1:'Jan', 2:'Fév', 3:'Mar', 4:'Avr', 5:'Mai', 6:'Jun',
               7:'Jul', 8:'Aoû', 9:'Sep', 10:'Oct', 11:'Nov', 12:'Déc'}
precip_mois['Mois_Label'] = precip_mois['Mois'].map(mois_labels)

fig2 = px.bar(
    precip_mois,
    x='Mois_Label', y='Precip_Total',
    title='🌧️ Précipitations Totales par Mois',
    color='Precip_Total',
    color_continuous_scale='Blues',
    text='Precip_Total'
)
fig2.update_traces(texttemplate='%{text:.0f}mm', textposition='outside')
fig2.update_layout(height=450)
fig2.show()

# ============================================
# GRAPHIQUE 3 — Répartition des alertes
# ============================================
alertes = df['severity_index'].value_counts().reset_index()
alertes.columns = ['Severity', 'Count']

colors = {'Rouge': '#e74c3c', 'Orange': '#e67e22', 
          'Jaune': '#f1c40f', 'RAS': '#2ecc71'}

fig3 = px.pie(
    alertes,
    values='Count', names='Severity',
    title='🚨 Répartition des Alertes Météo',
    color='Severity',
    color_discrete_map=colors,
    hole=0.4
)
fig3.update_layout(height=450)
fig3.show()

# ============================================
# GRAPHIQUE 4 — Indice de risque par zone
# ============================================
risque_zone = df.groupby('zone_geo')['indice_risque'].mean().round(2).reset_index()
risque_zone.columns = ['Zone', 'Indice_Risque']
risque_zone = risque_zone.sort_values('Indice_Risque', ascending=True)

fig4 = px.bar(
    risque_zone,
    x='Indice_Risque', y='Zone',
    title='⚠️ Indice de Risque Moyen par Zone',
    orientation='h',
    color='Indice_Risque',
    color_continuous_scale='RdYlGn_r',
    text='Indice_Risque'
)
fig4.update_traces(texttemplate='%{text}', textposition='outside')
fig4.update_layout(height=450)
fig4.show()

# ============================================
# GRAPHIQUE 5 — Evolution température par mois
# ============================================
temp_mois = df.groupby('mois').agg(
    Temp_Max=('temp_c', 'max'),
    Temp_Min=('temp_c', 'min'),
    Temp_Moy=('temp_c', 'mean')
).round(1).reset_index()
temp_mois['Mois_Label'] = temp_mois['mois'].map(mois_labels)

fig5 = go.Figure()
fig5.add_trace(go.Scatter(x=temp_mois['Mois_Label'], y=temp_mois['Temp_Max'],
                          name='Max', line=dict(color='red', width=2)))
fig5.add_trace(go.Scatter(x=temp_mois['Mois_Label'], y=temp_mois['Temp_Moy'],
                          name='Moyenne', line=dict(color='orange', width=2)))
fig5.add_trace(go.Scatter(x=temp_mois['Mois_Label'], y=temp_mois['Temp_Min'],
                          name='Min', line=dict(color='blue', width=2)))
fig5.update_layout(
    title='📈 Évolution des Températures par Mois',
    height=450,
    yaxis_title='Température (°C)',
    xaxis_title='Mois'
)
fig5.show()

# ============================================
# GRAPHIQUE 6 — Humidité vs Indice de risque
# ============================================
fig6 = px.scatter(
    df.sample(500),
    x='hum_pct', y='indice_risque',
    color='zone_geo',
    title='💧 Humidité vs Indice de Risque par Zone',
    labels={'hum_pct': 'Humidité (%)', 'indice_risque': 'Indice de Risque'},
    opacity=0.7
)
fig6.update_layout(height=450)
fig6.show()

print("\n🎉 Dashboard complet !")

📊 DASHBOARD — Agriculture-Résilience 2030
📍 Stations actives      : 20
📋 Total relevés         : 4626
🌡️  Température max       : 45.0°C
🚨 Alertes Rouge         : 39
⚠️  Indice risque moyen   : 51.12



🎉 Dashboard complet !


In [6]:
# Renommer severity_index_x en severity_index
df = df.rename(columns={'severity_index_x': 'severity_index'})
df = df.drop(columns=['severity_index_y'])

print("✅ Colonnes corrigées")
print(df.columns.tolist())

KeyError: "['severity_index_y'] not found in axis"